<a href="https://colab.research.google.com/github/parviza9999/Credit-Card-Fraud-Detection-/blob/main/notebooks/01_complete_eda_credit_card_fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Import libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Load dataset

In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Interview Node/Capstone_Project/Fraud Detection GCP Capstone/data/raw/creditcard.csv"

df = pd.read_csv(file_path)
df.head()

# Dataset shape

In [ ]:
df.shape

In [ ]:
df["Class"].value_counts()

In [ ]:
df["Class"].value_counts(normalize=True) * 100

# Data types and structure

In [ ]:
df.info()

# Missing values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

missing_summary.sort_values("missing_count", ascending=False)

# Duplicate records

In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percent = duplicate_count / len(df) * 100

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", duplicate_percent)

# Class distribution

In [ ]:
class_counts = df["Class"].value_counts().rename(index={0: "Non-Fraud", 1: "Fraud"})
class_percentages = df["Class"].value_counts(normalize=True).rename(index={0: "Non-Fraud", 1: "Fraud"}) * 100

class_distribution = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages
})

class_distribution

# Class imbalance bar chart

In [ ]:
plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar")
plt.title("Class Distribution: Fraud vs Non-Fraud")
plt.xlabel("Class")
plt.ylabel("Transaction Count")
plt.xticks(rotation=0)
plt.show()

# Fraud percentage statement

In [ ]:
fraud_count = df[df["Class"] == 1].shape[0]
non_fraud_count = df[df["Class"] == 0].shape[0]
fraud_percent = fraud_count / len(df) * 100

print(f"Fraud transactions: {fraud_count}")
print(f"Non-fraud transactions: {non_fraud_count}")
print(f"Fraud percentage: {fraud_percent:.4f}%")

# Descriptive statistics

In [ ]:
df.describe().T

# Amount summary by class

In [ ]:
amount_by_class = df.groupby("Class")["Amount"].describe()
amount_by_class

# Amount distribution

In [ ]:
plt.figure(figsize=(8, 4))
df["Amount"].plot(kind="hist", bins=100)
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.show()

# Log amount distribution

In [ ]:
df["Log_Amount"] = np.log1p(df["Amount"])

plt.figure(figsize=(8, 4))
df["Log_Amount"].plot(kind="hist", bins=100)
plt.title("Log-Transformed Transaction Amount Distribution")
plt.xlabel("log(Amount + 1)")
plt.ylabel("Frequency")
plt.show()

# Time distribution

In [ ]:
plt.figure(figsize=(8, 4))
df["Time"].plot(kind="hist", bins=100)
plt.title("Transaction Time Distribution")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.show()

# Correlation matrix

In [ ]:
corr_matrix = df.drop(columns=["Log_Amount"]).corr()

corr_matrix

# Correlation heatmap

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Matrix Heatmap")
plt.show()

# Correlation with target variable

In [ ]:
target_corr = corr_matrix["Class"].drop("Class").sort_values(key=abs, ascending=False)

target_corr

# Top correlations with fraud

In [ ]:
top_target_corr = target_corr.head(15)

plt.figure(figsize=(8, 5))
top_target_corr.sort_values().plot(kind="barh")
plt.title("Top Feature Correlations with Fraud Class")
plt.xlabel("Correlation with Class")
plt.ylabel("Feature")
plt.show()

# Multicollinearity analysis

For this dataset, V1 through V28 are PCA-transformed features, so most of them should not be strongly collinear. However, Time and Amount may behave differently.
### Find highly correlated feature pairs

In [ ]:
features_only = df.drop(columns=["Class", "Log_Amount"])

feature_corr = features_only.corr().abs()

upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_correlation"})
    .sort_values("abs_correlation", ascending=False)
)

high_corr_pairs.head(20)

# Filter strong multicollinearity

In [ ]:
strong_corr_pairs = high_corr_pairs[high_corr_pairs["abs_correlation"] >= 0.80]

strong_corr_pairs

## Multicollinearity Discussion

The multicollinearity review identifies whether any independent variables are strongly correlated with each other. Strong feature-to-feature correlation can affect linear models and feature interpretation. Because most variables in this dataset are PCA-transformed components, high multicollinearity is not expected among V1–V28. However, this check is included to document feature relationships before modeling.

# VIF analysis
Variance Inflation Factor (VIF) is a statistical metric used to detect and quantify the severity of multicollinearity (correlation) among independent variables in a multiple regression model. It measures how much the variance of an estimated regression coefficient is inflated due to collinearity

For any given independent variable, VIF is calculated as:VIF_*i* = 1/(1-R_*i* ^2)
Where:R_*i* ^2 is the coefficient of determination when variable *i* is regressed against all other independent variables in the model.

How to Interpret VIF ValuesVIF values start at 1, which indicates zero correlation between a given variable and the remaining predictors\
VIF = 1: No multicollinearity.\
1 < VIF <= 5: Low to moderate multicollinearity.\
VIF > 5 or 10: High multicollinearity. \
Variables with VIFs in this range generally warrant further investigation or removal from the model\
Why High VIF Matters: When VIF is high, the standard errors of the affected coefficients balloon, causing the coefficient estimates to become highly sensitive to minor changes in the dataset. This makes it difficult—if not impossible—to isolate the precise individual effect of each predictor on the dependent variable.

Variance Inflation Factor can be computationally heavier, but this dataset is manageable.

In [ ]:
vif_features = df.drop(columns=["Class", "Log_Amount"]).copy()

# Scale before VIF calculation
scaler = StandardScaler()
vif_scaled = scaler.fit_transform(vif_features)

vif_df = pd.DataFrame()
vif_df["feature"] = vif_features.columns
vif_df["VIF"] = [
    variance_inflation_factor(vif_scaled, i)
    for i in range(vif_scaled.shape[1])
]

vif_df.sort_values("VIF", ascending=False)

## VIF Interpretation

Variance Inflation Factor was used to check whether any feature can be strongly explained by other features. A high VIF may indicate multicollinearity. Since V1–V28 are PCA-transformed variables, high VIF values are not expected for those features. If Time or Amount show elevated VIF, those features should be reviewed during model development.

# Amount outlier review

In [ ]:
amount_q1 = df["Amount"].quantile(0.25)
amount_q3 = df["Amount"].quantile(0.75)
amount_iqr = amount_q3 - amount_q1

lower_bound = amount_q1 - 1.5 * amount_iqr
upper_bound = amount_q3 + 1.5 * amount_iqr

amount_outliers = df[(df["Amount"] < lower_bound) | (df["Amount"] > upper_bound)]

print("Amount Q1:", amount_q1)
print("Amount Q3:", amount_q3)
print("Amount IQR:", amount_iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Amount outlier count:", amount_outliers.shape[0])
print("Amount outlier percentage:", amount_outliers.shape[0] / len(df) * 100)

In [ ]:
# Amount outlier visualization: boxplot

plt.figure(figsize=(10, 4))
plt.boxplot(df["Amount"], vert=False, showfliers=True)
plt.title("Transaction Amount Outlier Review")
plt.xlabel("Transaction Amount")

# Add quartile markers
amount_q1 = df["Amount"].quantile(0.25)
amount_median = df["Amount"].quantile(0.50)
amount_q3 = df["Amount"].quantile(0.75)

plt.axvline(amount_q1, color='red', linestyle=':', label=f'Q1: {amount_q1:,.2f}', linewidth=2)
plt.axvline(amount_median, color='green', linestyle='--', label=f'Median (Q2): {amount_median:,.2f}', linewidth=2)
plt.axvline(amount_q3, color='purple', linestyle=':', label=f'Q3: {amount_q3:,.2f}', linewidth=2)

# Set x-axis limits to zoom in around the quartiles for better visibility
plt.xlim(amount_q1 - (amount_q3 - amount_q1) * 0.5, amount_q3 + (amount_q3 - amount_q1) * 0.5) # Adjust range as needed

plt.legend()
plt.show()

In [ ]:
# Amount distribution with IQR outlier threshold

plt.figure(figsize=(10, 5))
plt.hist(df["Amount"], bins=100)
plt.axvline(upper_bound, linestyle="--", linewidth=2, label=f"Upper IQR Bound: {upper_bound:.2f}")
plt.title("Transaction Amount Distribution with Outlier Threshold")
plt.xlabel("Transaction Amount")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Log-transformed amount distribution with outliers identified

df["Log_Amount"] = np.log1p(df["Amount"])
log_upper_bound = np.log1p(upper_bound)

plt.figure(figsize=(10, 5))
plt.hist(df["Log_Amount"], bins=100)
plt.axvline(log_upper_bound, linestyle="--", linewidth=2, label=f"Log Upper IQR Bound: {log_upper_bound:.2f}")
plt.title("Log-Transformed Transaction Amount Distribution")
plt.xlabel("log(Amount + 1)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
amount_outlier_fraud_rate = amount_outliers["Class"].mean() * 100

print(f"Fraud rate among amount outliers: {amount_outlier_fraud_rate:.4f}%")

## Amount Outlier Interpretation

The transaction amount distribution is highly right-skewed, meaning most transactions are relatively small while a smaller number of transactions have much larger values. The IQR method was used to identify unusually large transaction amounts.

These outliers should not be removed automatically because extreme transaction amounts may contain meaningful fraud signals. Instead, the outlier behavior will be documented and considered during preprocessing and model development.

# EDA summary table

In [ ]:
eda_summary = {
    "total_rows": df.shape[0],
    "total_columns": df.shape[1],
    "fraud_count": fraud_count,
    "non_fraud_count": non_fraud_count,
    "fraud_percent": fraud_percent,
    "total_missing_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(duplicate_count),
    "amount_outlier_count": int(amount_outliers.shape[0])
}

eda_summary_df = pd.DataFrame([eda_summary])
eda_summary_df

# Export EDA results to reports in Google Drive

In [ ]:
output_folder = "/content/drive/MyDrive/Interview Node/Capstone_Project/Fraud Detection GCP Capstone/reports"

eda_summary_df.to_csv(f"{output_folder}/eda_summary.csv", index=False)
target_corr.to_csv(f"{output_folder}/target_correlations.csv")
high_corr_pairs.to_csv(f"{output_folder}/feature_correlation_pairs.csv", index=False)
vif_df.to_csv(f"{output_folder}/vif_results.csv", index=False)

print("EDA report files saved.")

# EDA Findings Summary

The dataset contains credit card transactions with a highly imbalanced target variable. Fraud cases represent a very small fraction of the total dataset, which means accuracy alone is not an appropriate performance metric.

The EDA reviewed dataset structure, missing values, duplicate records, transaction amount distribution, time distribution, class imbalance, feature correlations, feature-to-feature collinearity, and Variance Inflation Factor.

Key modeling implications:

1. The model must be evaluated using precision, recall, F1-score, ROC-AUC, PR-AUC, and confusion matrix analysis.
2. Stratified train/test splitting is required to preserve the fraud ratio.
3. Scaling should be fitted only on the training data to avoid data leakage.
4. Class imbalance must be handled using appropriate methods such as class weighting, anomaly detection, threshold tuning, or resampling.
5. Correlation and VIF results should be documented, but PCA-transformed features V1–V28 are expected to have limited multicollinearity.
6. Outlier behavior in transaction amount should be reviewed but not automatically removed because extreme values may contain important fraud signals.